# Run the nycped analysis on Colab

Mounts Drive, installs the package, rebuilds the site table and writes the paper's tables and figure. Data files live in `MyDrive/TDSP/nyc_walks`. Use a fresh runtime, not the TDSP notebook.

While the repository is private, add a GitHub access token as a Colab secret named `GITHUB_TOKEN` (key icon in the left panel). Do not paste it into a cell.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/TDSP/nyc_walks'
os.makedirs(DATA, exist_ok=True)
os.environ['NYCPED_DATA_DIR'] = DATA
for f in ['tract_table.csv', 'ped_harm_crashes.csv', 'tract_poverty_moe.csv']:
    print(f, 'found' if os.path.exists(f'{DATA}/{f}') else 'MISSING')

In [ ]:
# Install the package. REF is the branch or tag. Use main after the paper branch is merged.
REF = 'paper-draft'
try:
    from google.colab import userdata
    key = userdata.get('GITHUB_TOKEN')
except Exception:
    key = None
url = 'git+https://' + (key + '@' if key else '') + f'github.com/AdiBak/nyc-pedestrian-poverty-exposure.git@{REF}'
!pip install -q "$url"
!python -m nycped --help

## 1. Download the public sources

Writes `MANIFEST.json` with the URL, date, size and checksum of each file.

In [ ]:
!python -m nycped fetch

## 2. Rebuild the site table

Takes a few minutes. Expect about 38,477 sites and 110,603 crosswalk segments. The table now carries the truck route column.

In [ ]:
!python -m nycped sites --crashes "$DATA/ped_harm_crashes.csv" --tract-table "$DATA/tract_table.csv"
!python -m nycped verify --sites "$DATA/site_dataset.csv"

## 3. Models, robustness, scale and reviewer checks

In [ ]:
!python -m nycped models --sites "$DATA/site_dataset.csv" --bootstrap 500

In [ ]:
!python -m nycped robustness --sites "$DATA/site_dataset.csv"

In [ ]:
!python -m nycped scale --sites "$DATA/site_dataset.csv"

In [ ]:
!python -m nycped checks --sites "$DATA/site_dataset.csv" --poverty-moe "$DATA/tract_poverty_moe.csv"

## 4. Paper tables, figure and numbers

Rebuilds the site table four more times for the radius and grouping checks, so allow several minutes.

In [ ]:
!python -m nycped paper --sites "$DATA/site_dataset.csv" --poverty-moe "$DATA/tract_poverty_moe.csv" --crashes "$DATA/ped_harm_crashes.csv" --tract-table "$DATA/tract_table.csv" --out "$DATA/paper_generated"

In [ ]:
# Zip the outputs and download them. Also left in the Drive folder.
!cd "$DATA" && zip -qr paper_generated.zip paper_generated
from google.colab import files
files.download(f'{DATA}/paper_generated.zip')